# Corrélation entre anguille et spectre de vagues WW3

In [ ]:
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import holoviews as hv
import numpy as np
import time as temps
from datetime import datetime
import fsspec
import matplotlib.dates as mdates

###########
# Usual libraries
###########
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import dates
import math
import netCDF4 as NC4
import os
import datetime
import pandas as pd

###########
# Signal processing functions
###########
from scipy.signal import detrend, welch

###########
# Additionals functions
###########
from scipy.io import loadmat
from scipy.stats.distributions import chi2
import numpy.matlib
from IPython import display

## Définition des fonctions utilisées

**Lissage logarithmique sur les données de pression des anguilles**

In [ ]:
from scipy.fft import rfft, rfftfreq
from scipy.interpolate import interp1d
from scipy.signal import fftconvolve


def log_smooth_spectrum(x, fs, n_log=1000, fmin=0.01, fmax=1, sigma_oct=0.05):
    """
    Lissage logarithmique du spectre de puissance.

    Paramètres
    ----------
    x : ndarray
        Signal temporel (1D).
    fs : float
        Fréquence d'échantillonnage en Hz.
    n_log : int
        Nombre de points sur l'axe log pour l'interpolation.
    fmin : float
        Fréquence minimale (éviter f=0 pour log).
    fmax : float ou None
        Fréquence maximale (Nyquist si None).
    sigma_oct : float
        Largeur de la gaussienne de lissage en octaves
        (ex: 0.25 = 1/4 d’octave).

    Renvoie
    -------
    freqs : ndarray
        Axe des fréquences linéaire (FFT).
    P : ndarray
        Spectre de puissance non lissé (|S|^2).
    P_smooth : ndarray
        Spectre de puissance lissé (même axe `freqs`).
    """

    if fmax is None:
        fmax = fs / 2.0

    # FFT positive
    S = rfft(x)
    freqs = rfftfreq(len(x), 1 / fs)

    # Spectre de puissance
    P = np.abs(S) ** 2

    # Restriction [fmin, fmax]
    mask = (freqs >= fmin) & (freqs <= fmax)
    freqs_cut = freqs[mask]
    P_cut = P[mask]

    # Axe log
    logf = np.linspace(np.log(freqs_cut[0]), np.log(freqs_cut[-1]), n_log)
    f_log = np.exp(logf)

    # Interpolation du spectre sur l’axe log
    interp = interp1d(
        freqs_cut, P_cut, kind="linear", bounds_error=False, fill_value="extrapolate"
    )
    P_log = interp(f_log)

    # Noyau gaussien (sigma en octaves)
    sigma_ln = sigma_oct * np.log(2)  # conversion octaves → log naturel
    half = int(4 * (sigma_ln / (logf[1] - logf[0])))  # demi-taille noyau
    t = np.arange(-half, half + 1)
    kernel = np.exp(-0.5 * (t * (logf[1] - logf[0]) / sigma_ln) ** 2)
    kernel /= kernel.sum()

    # Convolution dans l'axe log
    P_log_smooth = fftconvolve(P_log, kernel, mode="same")

    # Retour vers axe linéaire
    interp_back = interp1d(
        f_log, P_log_smooth, kind="linear", bounds_error=False, fill_value="extrapolate"
    )
    P_smooth = interp_back(freqs_cut)

    return freqs_cut, P_cut, P_smooth

**Fonctions annexes pour le calcul**

In [ ]:
def stations_proches(ds, lat, lon, delta_lat=0.2, delta_lon=0.2):
    lat_min, lat_max = lat - delta_lat, lat + delta_lat
    lon_min, lon_max = lon - delta_lon, lon + delta_lon

    mask_nodes = (
        (ds["latitude"] >= lat_min)
        & (ds["latitude"] <= lat_max)
        & (ds["longitude"] >= lon_min)
        & (ds["longitude"] <= lon_max)
    )
    index_nodes = ds.node.values[mask_nodes]  # Les vrais IDs

    nodes = ds.sel(node=index_nodes)  # Sélection par valeur, pas par position
    return nodes, index_nodes


def station_proche(ds, lat, lon):
    dlat = ds["latitude"] - lat
    dlon = ds["longitude"] - lon

    # distance approximative en degrés (valable localement)
    dist2 = dlat**2 + dlon**2

    idx = dist2.argmin().item()

    return ds.isel(node=idx)


def conv_time(date_debut, date_evt, pas, step="seconds"):

    # Calcul de la différence de temps
    diff_t = date_evt - date_debut
    # Conversion en secondes (ou autre unité selon step)
    if step == "seconds":
        diff_unit = diff_t.total_seconds()
    elif step == "minutes":
        diff_unit = diff_t.total_seconds() / 60
    elif step == "hours":
        diff_unit = diff_t.total_seconds() / 3600
    elif step == "days":
        diff_unit = diff_t.total_seconds() / (3600 * 24)
    else:
        raise ValueError(
            "Unité de temps non supportée. Choisir 'seconds', 'minutes', 'hours' ou 'days'"
        )

    # Calcul de l'indice
    indice = int(diff_unit / pas)

    return indice

**Calcul de la différence, entre le spectre de l'anguille, et le spectre d'élévation des vagues.**

In [ ]:
def calcul_diff(spec_fish, ef_WW3, freq_fish, freq_wave):
    """
    Compare le spectre poisson (spec_fish) et WW3 (ef_WW3)
    en les ramenant sur la même grille fréquentielle.

    spec_fish : spectre poisson (array)
    ef_WW3    : spectre WW3 en log10 (array)
    freq_fish : fréquences poisson (array)
    freq_wave : fréquences WW3 (array)
    """

    # Conversion WW3 log10 -> linéaire
    spec_WW3 = np.power(10, ef_WW3) - 1e-12

    # On garde uniquement la bande commune [fmin, fmax]
    fmin, fmax = freq_fish.min(), freq_fish.max()
    mask = (freq_wave >= fmin) & (freq_wave <= fmax)

    freq_wave_common = freq_wave[mask]
    spec_WW3_common = spec_WW3[mask]

    # Interpolation du spectre poisson sur la grille WW3
    spec_fish_interp = np.interp(freq_wave_common, freq_fish, spec_fish)

    # Différence point à point
    diff = np.abs(spec_fish_interp - spec_WW3_common)

    return diff, freq_wave_common


def dispNewtonTH(f, dep):
    # This function inverts the linear dispersion relation (2*pi*f)^2=g*k*tanh(k*dep) to get
    # k from f and dep. 2 Arguments: f and dep are frequency in Hertz and depth in meters
    # % Uses Newton method. Original code from T.H.C. Herbers
    eps = 0.000001
    g = 9.81
    sig = 2 * np.pi * f
    Y = dep * sig**2.0 / g

    X = np.sqrt(Y)
    I = 1
    F = 1
    while abs(np.amax(F)) > eps:
        H = np.tanh(X)
        F = Y - X * H
        FD = -H - X / (np.cosh(X) ** 2)
        X = X - F / FD

    dispNewtonTH = X / dep

    return dispNewtonTH
    # return 2*np.pi*f/np.sqrt(9.81*dep)

**Fonction principale** ,à partir d'un ID de tag, on utilise la géolocalisation déjà calculé. Pour chaque point de géoloc, on utlise les stations WW3 autour et on cherche une corrélation entre les scpectres de vagues et ceux des poissons. 

In [ ]:
def spectre_via_geoloc(df_loc, ds_anguille, ID, ds_WW3):
    # Normalisation de l'ID
    if isinstance(ID, str):
        if ID.startswith("A"):
            ID = int(ID[1:])  # enlève le A et convertit en entier
        else:
            ID = int(ID)  # convertit une chaîne "15789" en entier

    # Vérif debug
    print("ID après nettoyage:", ID)
    print("IDs disponibles dans ds_loc:", ds_loc.ID.values[:10])

    # Sélection sécurisée
    # if ID not in ds_loc.ID.values:
    #   raise ValueError(f"ID '{ID}' non trouvé dans ds_loc")

    # Sélection des données pour ce tag
    subset = ds_loc.sel(ID=ID)

    # Conversion en DataFrame pour manipuler facilement
    df = (
        subset.to_dataframe()[["lat", "lon"]]
        .reset_index()
        .dropna(subset=["lat", "lon"])
    )
    # Construire les trios (time, lat, lon)
    trios = list(zip(df["time"], df["lat"], df["lon"]))
    date_debut = data["datetime"].values[0]
    N = 600  # nombre de points (= 20 minutes)
    dt = 2  # pas de temps en secondes
    freq_WW3 = ds_WW3["f"].values

    dico_total = {}  # Stockage des différence entre les spectres

    start_time = temps.time()

    for i in range(len(trios)):

        time, lat, lon = trios[i][0], trios[i][1], trios[i][2]
        data_WW3_time = ds_WW3.sel(time=time, method="nearest")
        # print(date_debut,time)
        indice = conv_time(date_debut, time, dt, step="seconds")
        segment = data["pressure"].values[indice : indice + N]

        dico_diff = {}

        # print(f"Indice={indice}, longueur totale={len(data['pressure'])}, segment={len(segment)}")

        freqs2, P2, P_smooth = log_smooth_spectrum(
            segment, 0.5, n_log=1000, fmin=0.03, fmax=0.953, sigma_oct=0.1
        )
        liste_node, index_nodes = stations_proches(data_WW3_time, lat, lon)
        # print("Les valeurs des node:", liste_node.node.values)
        # if i < 3:
        # print(liste_node)
        values_ef = liste_node["ef"].values

        # Transformer en une liste de spectres (un par nœud)
        list_spectres = [values_ef[:, i].tolist() for i in range(values_ef.shape[1])]

        # print("ICIIIIIIIIIIIIIIIIIIIIIII", list_spectres)
        for j in range(len(list_spectres)):
            # print(len(values_ef),len(values_ef[i]),print(len(freq_WW3)))
            # print((values_ef),(values_ef[i]))
            diff, freq_diff = calcul_diff(P_smooth, list_spectres[j], freqs2, freq_WW3)
            dico_diff[index_nodes[j]] = [diff, freq_diff]

            # Afficher l'avancement + temps écoulé
            elapsed = temps.time() - start_time
            # print(f"[{i+1}/{len(trios)}] Temps = {time}, Nœud {j+1}/{len(list_spectres)} traité | {elapsed:.2f}s écoulées")

        dico_total[time] = dico_diff
    return dico_total

Calcul des corrélations les plus importantes dans la liste des stations.

In [ ]:
def mini_diff_day(data_day, top_n=5):
    temp = []
    for node_id, values in data_day.items():
        # chaque values est une liste de tuples (diff_array, freq_array)

        diff, freq = values[0], values[1]
        total = np.sum(diff)
        temp.append((node_id, total, diff))
    # Trier et garder les N plus faibles
    temp_sorted = sorted(temp, key=lambda x: x[1])[:top_n]
    return temp_sorted

## Ouverture des données WW3 de 2023 

In [ ]:
url = "https://data-dataref.ifremer.fr/ww3/resourcecode/HINDCAST/2019/12/FIELD_NC/RSCD_WW3-RSCD-UG_201912_ef.nc"
file = fsspec.open(url).open()

ds = xr.open_dataset(file)

lat = ds["latitude"]
lon = ds["longitude"]

# définir les bornes
lat_min, lat_max = 50, 51.3
lon_min, lon_max = 0, 3

# construire un masque sur les nodes
mask_nodes = (lat >= lat_min) & (lat <= lat_max) & (lon >= lon_min) & (lon <= lon_max)

# sélectionner seulement les nodes valides
ds_filtered = ds.isel(node=mask_nodes)
(ds_filtered["ef"])

ds

## Ouverture de géoloc des anguilles

In [ ]:
# Charger le CSV
df_loc = pd.read_csv(
    "/home/jovyan/pangeo-fish/notebooks/nouveau_tag/geolocalisation_anguille.csv",
    dtype=str,
)


# Fonction pour inverser jour et mois si jour <= 12
def fix_day_month(date_str):
    try:
        day, month, year = map(int, date_str.split("/"))
        if day <= 12:
            # Inverser seulement si ça semble problématique
            day, month = month, day
        return f"{day:02d}/{month:02d}/{year}"
    except:
        return date_str


# Appliquer la correction
df_loc["Date"] = df_loc["Date"].apply(fix_day_month)

# Convertir en datetime
df_loc["Date"] = pd.to_datetime(df_loc["Date"], dayfirst=True, errors="coerce")

# Convertir ID, lat et lon en numériques
df_loc["ID"] = df_loc["ID"].astype(int)
df_loc["MPL.Avg.Lat"] = pd.to_numeric(df_loc["MPL.Avg.Lat"], errors="coerce")
df_loc["MPL.Avg.Lon"] = pd.to_numeric(df_loc["MPL.Avg.Lon"], errors="coerce")

# Renommer colonnes pour xarray
df_loc = df_loc.rename(
    columns={"Date": "time", "MPL.Avg.Lat": "lat", "MPL.Avg.Lon": "lon"}
)

# Créer le Dataset
import xarray as xr

ds_loc = xr.Dataset.from_dataframe(df_loc.set_index(["ID", "time"]))


# Vérif rapide
ds_loc

## Ouverture des données d'anguilles 

In [ ]:
# Lire le CSV avec pandas
df = pd.read_csv(
    "/home/jovyan/pangeo-fish/notebooks/nouveau_tag/A15789/fichier_base.csv"
)
print(df)
data = xr.Dataset.from_dataframe(
    df.set_index(["datetime"])
)  # si tu as une colonne 'tim
# Convertir en xarray Dataset


# 2️⃣ Convertir la colonne temps en datetime
# Remplace 'nom_de_la_colonne_temps' par le vrai nom de ta colonne
df["datetime"] = pd.to_datetime(df["datetime"])

# 3️⃣ Créer un Dataset Xarray avec le temps comme index
data = xr.Dataset.from_dataframe(df.set_index("datetime"))
print(df)

In [ ]:
data

In [ ]:
import hvplot.xarray  # active le support xarray
import holoviews as hv

hv.extension("bokeh")  # ⚠️ nécessaire pour afficher les graphiques

import s3fs
import pandas as pd

fs = s3fs.S3FileSystem(anon=False)  # anon=False = accès authentifié

path = "s3://gfts-ifremer/eel/tag/raw/A15700.csv"
with fs.open(path, "r") as f:
    df = pd.read_csv(f)


# 2️⃣ Convertir la colonne 'datetime' en format temporel
df["datetime"] = pd.to_datetime(df["datetime"])

# 3️⃣ Créer un Dataset xarray avec le temps comme index
data = xr.Dataset.from_dataframe(df.set_index("datetime"))

# 4️⃣ Vérification rapide
print(data)

In [ ]:
# 5️⃣ Afficher la variable de pression avec rasterisation
plot = data["pressure"].hvplot(rasterize=True, title="Pression dans le temps")
plot

In [ ]:
data["pressure"].plot()

In [ ]:
data["pressure"].sel(datetime=slice("2018-12-07 6:00:00", "2019-01-07 9:00:00")).plot()

In [ ]:
date_debut = pd.Timestamp("2019-12-17 10:00:00")
date = pd.Timestamp("2020-01-20 09:00:00")
two_days = int(1 / 2 * 3600 * 9.5 * 1)
N = two_days
indice_ex = conv_time(date_debut, date, 2)
exemple = np.array(data["pressure"].values[indice_ex : indice_ex + two_days])
plt.plot(exemple)

In [ ]:
import numpy as np


def _moving_range(arr, w):
    """Renvoie un tableau de longueur n-w+1 où chaque élément est max-min sur la fenêtre arr[i:i+w]."""
    n = len(arr)
    if w <= 0 or w > n:
        return np.array([])  # pas de fenêtre valide
    try:
        from numpy.lib.stride_tricks import sliding_window_view

        sw = sliding_window_view(arr, w)  # shape (n-w+1, w)
        return sw.max(axis=1) - sw.min(axis=1)
    except Exception:
        # fallback (moins efficace)
        out = np.empty(n - w + 1, dtype=float)
        for i in range(n - w + 1):
            slice_ = arr[i : i + w]
            out[i] = slice_.max() - slice_.min()
        return out


def detect_and_align_jumps(
    data,
    fs=0.5,  # Hz (0.5 => 1 mesure toutes les 2 s)
    jump_window_minutes=10,  # fenêtre pour détectefr le saut (minutes)
    jump_threshold=20.0,  # seuil (mètres) pour considérer un saut
    activity_window_minutes=5,  # fenêtre pour estimer activité / variance (minutes)
    activity_threshold=15.0,  # seuil (mètres) : si range > threshold => actif
    calm_window_minutes=15,  # fenêtre pour vérifier "calme" avant/après (minutes)
    calm_threshold=15.0,  # si range <= threshold => calme
    avg_window_minutes=15,  # fenêtre utilisée pour calculer moyennes avant/après (minutes)
    fill_method="linear",  # "nan" ou "linear"
    align_to="before",  # si les deux côtés sont calmes: "before" (par défaut) ou "after"
):
    """
    Retourne (corrected, jumps_info)
    - corrected : array float de même taille que data (avec corrections)
    - jumps_info : liste de dicts {start_idx, end_idx, mean_before, mean_after, diff, calm_before, calm_after}
    """
    data = np.asarray(data, dtype=float)
    n = len(data)
    if n == 0:
        return data, []

    # convertir minutes -> échantillons
    jump_w = max(1, int(round(jump_window_minutes * 60 * fs)))
    activity_w = max(1, int(round(activity_window_minutes * 60 * fs)))
    calm_w = max(1, int(round(calm_window_minutes * 60 * fs)))
    avg_w = max(1, int(round(avg_window_minutes * 60 * fs)))

    # indice maximum pour construction de moving ranges
    ranges_activity = _moving_range(data, activity_w)  # length m = n - activity_w + 1
    m = len(ranges_activity)

    corrected = data.copy()  # on applique les corrections ici (séquentiellement)
    jumps_info = []

    i = 0
    while i <= n - jump_w - 1:
        # Détection brute du jump (net change sur la fenêtre jump_w)
        start_val = data[i]
        end_val = data[i + jump_w]
        if abs(end_val - start_val) > jump_threshold:
            # trouver les indices de fenêtres d'activité (j) qui se chevauchent la zone candidate
            j_low = max(0, i - (activity_w - 1))
            j_high = min(m - 1, i + jump_w - 1)
            if j_low > j_high:
                # aucun j valide (fenêtre activity trop grande pour rester dans les bords)
                # fallback: on prend l'intervalle simple
                start_idx = i
                end_idx = i + jump_w
            else:
                # windows actifs sur la plage
                j_range_idx = np.arange(m)
                overlapping = (j_range_idx >= j_low) & (j_range_idx <= j_high)
                active_mask = (ranges_activity > activity_threshold) & overlapping

                if not np.any(active_mask):
                    # pas d'activité suffisamment forte dans les windows qui chevauchent
                    # on fallback au sweep simple
                    start_idx = i
                    end_idx = i + jump_w
                else:
                    # on récupère le bloc contigu de windows "actives" qui chevauche la zone
                    active_indices = np.where(ranges_activity > activity_threshold)[0]
                    # prendre l'intersection minimale/ max qui touche la zone
                    hit_js = np.where(active_mask)[0]
                    j_start = hit_js.min()
                    j_end = hit_js.max()
                    # étendre le bloc tant que les fenêtres adjacentes sont actives
                    while (
                        j_start > 0
                        and ranges_activity[j_start - 1] > activity_threshold
                    ):
                        j_start -= 1
                    while (
                        j_end < m - 1
                        and ranges_activity[j_end + 1] > activity_threshold
                    ):
                        j_end += 1
                    # Convertir indices de fenêtre en indices d'échantillons (fenêtre j couvre samples j..j+activity_w-1)
                    start_idx = j_start
                    end_idx = j_end + activity_w - 1
                    # clamp
                    start_idx = max(0, start_idx)
                    end_idx = min(n - 1, end_idx)

            # maintenant vérifier calme avant/après (sur calm_w échantillons)
            before_slice_start = max(0, start_idx - calm_w)
            before_slice = (
                data[before_slice_start:start_idx] if start_idx > 0 else np.array([])
            )
            after_slice_end = min(n, end_idx + 1 + calm_w)
            after_slice = (
                data[end_idx + 1 : after_slice_end] if end_idx + 1 < n else np.array([])
            )

            calm_before = (len(before_slice) >= 1) and (
                (np.max(before_slice) - np.min(before_slice)) <= calm_threshold
            )
            calm_after = (len(after_slice) >= 1) and (
                (np.max(after_slice) - np.min(after_slice)) <= calm_threshold
            )

            # on n'applique la correction que si au moins un côté est calme
            if calm_before or calm_after:
                # moyennes utilisées pour l'alignement (prendre sur corrected pour chaîner les corrections)
                # moyen_before : avg_w points immédiatement avant start_idx (si possible)
                mb_start = max(0, start_idx - avg_w)
                mb = corrected[mb_start:start_idx] if start_idx > 0 else np.array([])
                if calm_before and len(mb) >= 1:
                    mean_before = np.nanmean(mb)
                elif calm_before and len(mb) == 0:
                    # fallback : utiliser la valeur juste avant si existe
                    mean_before = (
                        float(corrected[start_idx - 1]) if start_idx > 0 else np.nan
                    )
                else:
                    mean_before = np.nan

                ma_end = min(n, end_idx + 1 + avg_w)
                ma = (
                    corrected[end_idx + 1 : ma_end] if end_idx + 1 < n else np.array([])
                )
                if calm_after and len(ma) >= 1:
                    mean_after = np.nanmean(ma)
                elif calm_after and len(ma) == 0:
                    mean_after = (
                        float(corrected[end_idx + 1]) if end_idx + 1 < n else np.nan
                    )
                else:
                    mean_after = np.nan

                # si les deux sont calmes, choisir align_to
                if (
                    calm_before
                    and calm_after
                    and np.isfinite(mean_before)
                    and np.isfinite(mean_after)
                ):
                    if align_to == "before":
                        # on aligne la partie APRES sur la moyenne BEFORE
                        diff = mean_after - mean_before
                        corrected[end_idx + 1 :] = corrected[end_idx + 1 :] - diff
                    else:  # align_to == "after"
                        diff = mean_before - mean_after
                        corrected[:start_idx] = corrected[:start_idx] - diff
                else:
                    # cas où seul un côté est calme : on aligne le côté non-calme sur le côté calme
                    if calm_before and np.isfinite(mean_before):
                        # aligne la partie APRES sur mean_before
                        # calculer mean_after réel s'il existe sinon utiliser valeur immédiate
                        if np.isfinite(mean_after):
                            diff = mean_after - mean_before
                        else:
                            # estimate mean_after from raw value just after end_idx if exists
                            mean_after_est = (
                                float(corrected[end_idx + 1])
                                if end_idx + 1 < n
                                else 0.0
                            )
                            diff = mean_after_est - mean_before
                        corrected[end_idx + 1 :] = corrected[end_idx + 1 :] - diff
                    elif calm_after and np.isfinite(mean_after):
                        # aligne la partie BEFORE sur mean_after
                        if np.isfinite(mean_before):
                            diff = mean_before - mean_after
                        else:
                            mean_before_est = (
                                float(corrected[start_idx - 1])
                                if start_idx > 0
                                else 0.0
                            )
                            diff = mean_before_est - mean_after
                        corrected[:start_idx] = corrected[:start_idx] - diff

                # remplir la zone du saut selon fill_method (sur l'array corrigée)
                L = end_idx - start_idx + 1
                if L > 0:
                    if fill_method == "nan":
                        corrected[start_idx : end_idx + 1] = np.nan
                    elif fill_method == "linear":
                        # points de raccord (prendre valeurs corrigées juste avant/après si possible)
                        if start_idx > 0:
                            v0 = corrected[start_idx - 1]
                        else:
                            v0 = (
                                mean_before
                                if np.isfinite(mean_before)
                                else corrected[start_idx]
                            )
                        if end_idx + 1 < n:
                            v1 = corrected[end_idx + 1]
                        else:
                            v1 = (
                                mean_after
                                if np.isfinite(mean_after)
                                else corrected[end_idx]
                            )
                        corrected[start_idx : end_idx + 1] = np.linspace(v0, v1, L)
                    else:
                        # autre option possible : hold_before, hold_after
                        if fill_method == "hold_before" and start_idx > 0:
                            corrected[start_idx : end_idx + 1] = corrected[
                                start_idx - 1
                            ]
                        elif fill_method == "hold_after" and end_idx + 1 < n:
                            corrected[start_idx : end_idx + 1] = corrected[end_idx + 1]
                        else:
                            corrected[start_idx : end_idx + 1] = np.nan

                jumps_info.append(
                    {
                        "start_idx": int(start_idx),
                        "end_idx": int(end_idx),
                        "mean_before": (
                            float(mean_before) if np.isfinite(mean_before) else None
                        ),
                        "mean_after": (
                            float(mean_after) if np.isfinite(mean_after) else None
                        ),
                        "diff": float(
                            (mean_after - mean_before)
                            if np.isfinite(mean_after) and np.isfinite(mean_before)
                            else np.nan
                        ),
                        "calm_before": bool(calm_before),
                        "calm_after": bool(calm_after),
                    }
                )

                # sauter l'intervalle traité
                i = end_idx + 1
                continue  # passe au prochain i
            else:
                # ni avant ni après calme -> ignorer ce candidat
                i += 1
                continue
        else:
            i += 1

    return corrected, jumps_info


def detect_and_align_jumps_variance(
    data,
    fs=0.5,  # Hz
    segment_slice=None,  # tuple (start_idx, end_idx)
    var_window_minutes=5,  # fenêtre pour variance glissante (minutes)
    var_step_minutes=2,  # pas pour moyenne variance (minutes)
    jump_var_threshold=50.0,  # seuil de variance pour détecter un "jump"
    calm_var_window_minutes=10,  # fenêtre pour vérifier calme avant/après (minutes)
    calm_var_threshold=5.0,  # seuil variance faible = calme
    fill_method="linear",
):
    """
    Detect jumps using variance peaks, then align and optionally fill them.

    Returns:
    - corrected : array float (mêmes dimensions que segment)
    - jumps_info : list of dicts {start_idx, end_idx, var_peak, calm_before, calm_after}
    """
    # Extraire le segment
    if segment_slice is None:
        segment_da = data.pressure
    else:
        start, end = segment_slice
        segment_da = data.pressure.isel(datetime=slice(start, end))

    segment = segment_da.values
    n = len(segment)

    # Fenêtre variance glissante
    window_points = int(var_window_minutes * 60 * fs)  # points
    rolling_var = (
        pd.Series(segment).rolling(window=window_points, center=True).var().values
    )

    # Moyenne de variance sur step_points
    step_points = int(var_step_minutes * 60 * fs)
    n_windows = len(rolling_var) // step_points
    mean_var = np.array(
        [
            rolling_var[i * step_points : (i + 1) * step_points].mean()
            for i in range(n_windows)
        ]
    )

    # indices des fenêtres
    time_idx = np.array([i * step_points for i in range(n_windows)])

    corrected = segment.copy()
    jumps_info = []

    # Détection des pics de variance > jump_var_threshold
    for i, v in enumerate(mean_var):
        if v > jump_var_threshold:
            # fenêtre suspecte
            jump_start = time_idx[i]
            jump_end = min(jump_start + 10, n)  # max 10 points pour le jump

            # vérifier calme avant/après (calm_var_window_minutes)
            calm_window_points = int(calm_var_window_minutes * 60 * fs)
            before_start = max(0, jump_start - calm_window_points)
            after_end = min(n, jump_end + calm_window_points)

            calm_before = (
                np.nanvar(segment[before_start:jump_start]) <= calm_var_threshold
            )
            calm_after = np.nanvar(segment[jump_end:after_end]) <= calm_var_threshold

            # Appliquer correction si au moins un côté calme
            if calm_before or calm_after:
                # Raccord linéaire sur jump
                if jump_end > jump_start:
                    v0 = (
                        corrected[jump_start - 1]
                        if jump_start > 0
                        else corrected[jump_start]
                    )
                    v1 = (
                        corrected[jump_end] if jump_end < n else corrected[jump_end - 1]
                    )
                    corrected[jump_start:jump_end] = np.linspace(
                        v0, v1, jump_end - jump_start
                    )

            jumps_info.append(
                {
                    "start_idx": jump_start,
                    "end_idx": jump_end,
                    "var_peak": float(v),
                    "calm_before": bool(calm_before),
                    "calm_after": bool(calm_after),
                }
            )

    return corrected, jumps_info

In [ ]:
corrected_pressure, jumps_info = detect_and_align_jumps(exemple)
print(corrected_pressure)
plt.plot(corrected_pressure)

In [ ]:
offbot = 0.1  # height of pressure sensor over bottom [m]
zone = 0
fs = 0.5
nfft = 400  # nfft specifies the FFT length that csd uses.
df = fs / nfft  # Frequential resolution
numoverlap = (
    nfft / 2
)  # numoverlap is the number of samples by which the sections overlap.
Nf = int(nfft / 2 + 1)
Nfo = Nf - 1  # up to Nyquist.
# freq=np.linspace(df,df*(Nf-1),Nf-1)

print("Frequency resolution is", df, "Hz.")

# --- Paramètres
fs = 0.5  # Hz (à adapter)
nfft = 400
Nf = int(nfft / 2 + 1)
fmax = 0.08
dfreq = fs / nfft
ifmax = int(fmax / dfreq)

dep = 25.0  # profondeur réelle (m), à renseigner !
offbot = 0.1  # capteur à 10 cm au-dessus du fond

# --- Extraire un segment


# --- Spectral analysis
t = np.arange(len(corrected_pressure)) / fs
freq, E_p = welch(
    corrected_pressure,
    fs=fs,
    window="hann",
    nperseg=nfft,
    detrend="linear",
    return_onesided=True,
    scaling="density",
)

# --- Inversion dispersion (à coder si pas dispo)
k = dispNewtonTH(freq, dep)
k[0] = 0
M = np.cosh(k * dep) / np.cosh(k * offbot)
cosur = M**2
cosur[0] = 1.0
cosur[ifmax + 1 : Nf] = 1.0
Ef = E_p * cosur

# --- Plot
plt.figure(figsize=(12, 4))
plt.plot(t[:N], corrected_pressure[:N], "m-")
plt.xlabel("time (s)")
plt.ylabel("pressure (m water)")
plt.title("First 1000 samples of raw pressure data")
plt.show()

plt.figure(figsize=(12, 4))
plt.loglog(freq, E_p, "m-", label="Bottom pressure PSD")
plt.loglog(freq, Ef, "k-", label="Surface corrected_pressure PSD (corrected)")
plt.xlabel("Frequency (Hz)")
plt.ylabel("PSD (m²/Hz)")
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_spectrum(signal, fs):
    # Remplacer les NaN éventuels par la moyenne
    sig = np.copy(signal)
    sig[np.isnan(sig)] = np.nanmean(sig)

    n = len(sig)
    freqs = np.fft.rfftfreq(n, d=1 / fs)
    fft_vals = np.fft.rfft(sig - np.mean(sig))
    psd = np.abs(fft_vals) ** 2 / n

    plt.figure(figsize=(8, 5))
    plt.loglog(freqs[1:], psd[1:])  # on ignore f=0
    plt.xlabel("Fréquence (Hz)")
    plt.ylabel("Spectre de puissance")
    plt.title("Spectre en log-log")
    plt.grid(True, which="both", ls="--")
    plt.show()


# Exemple d'appel
fs = 0.5  # Hz (1 point toutes les 2 secondes)
plot_spectrum(corrected_pressure, fs)

In [ ]:
print(responses[1 / 12])
plt.plot(time, responses[1 / 12])
plt.show()
plt.plot(time, responses[1 / 10])
plt.show()

### Résoudre le problème de compression dynamique des données, annulation du peigne 

In [ ]:
pressure = data["pressure"].values
time = data["datetime"].values
date_debut = pd.Timestamp("2019-12-17 10:00:00")
date_evt = [
    pd.Timestamp("2020-01-03T10:00.00"),
    pd.Timestamp("2020-01-03T12:00.00"),
    pd.Timestamp("2020-01-18T15:00.00"),
    pd.Timestamp("2020-01-20T16:30.00"),
    pd.Timestamp("2020-01-22T12:30.00"),
    pd.Timestamp("2020-01-30T18:30.00"),
]
N = 500
pas = 2
fs = 1 / pas  # freq échantillonnage


# --------------------------------------------------------
# filtre de Gabor 1D
# --------------------------------------------------------
def gabor_filter(f0, sigma, fs, duration=60):
    """
    Crée un filtre de Gabor 1D (temps, fenêtre finie).
    f0 : fréquence centrale (Hz)
    sigma : largeur gaussienne (s)
    fs : fréquence d'échantillonnage
    duration : durée totale du filtre (s)
    """
    t = np.arange(-duration / 2, duration / 2, 1 / fs)
    g = np.exp(-(t**2) / (2 * sigma**2)) * np.exp(1j * 2 * np.pi * f0 * t)
    return t, g / np.linalg.norm(g)  # normalisation


# --------------------------------------------------------
# banque de filtres
# --------------------------------------------------------
periods = [6, 10, 14, 18]  # 5 filtres espacés
frequencies = [1.0 / T for T in periods]
# frequencies.append(0.071428)
print(frequencies)
filters = {}
for f0 in frequencies:
    T = 1 / f0
    sigma = T
    t, g = gabor_filter(f0, sigma, fs, duration=5 * T)
    filters[f0] = g

# --------------------------------------------------------
# convolution
# --------------------------------------------------------
responses = {}
for f0, g in filters.items():
    y = fftconvolve(pressure, g, mode="same")
    responses[f0] = np.abs(y)


indice_hist = [conv_time(date_debut, date, pas, step="seconds") for date in date_evt]
print(indice_hist)

In [ ]:
from scipy.signal import welch

for i in range(len(indice_hist)):
    # --- Extraire le segment
    seg_time = time[indice_hist[i] : indice_hist[i] + N]
    seg_pressure = pressure[indice_hist[i] : indice_hist[i] + N]

    # --- Création figure avec 3 sous-graphiques
    fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=False)

    # 1) Signal brut
    axes[0].plot(seg_time, seg_pressure, color="k", label="Signal brut")
    axes[0].set_ylabel("Amplitude")
    axes[0].set_title(f"Signal brut - Date: {date_evt[i]}")
    axes[0].legend()

    # 2) Réponses filtrées (tous les filtres Gabor sur série temporelle)
    for f0, y in responses.items():
        axes[1].plot(
            seg_time,
            y[indice_hist[i] : indice_hist[i] + N],
            label=f"Gabor {f0:.3f} Hz (~{1/f0:.1f}s)",
        )
    axes[1].set_xlabel("Temps")
    axes[1].set_ylabel("Amplitude")
    axes[1].set_title("Réponses Gabor (série temporelle)")
    axes[1].legend()

    # 3) Spectres de toutes les réponses Gabor
    for f0, y in responses.items():
        seg_response = y[indice_hist[i] : indice_hist[i] + N]
        freq, Pxx = welch(
            seg_response,
            fs=fs,
            window="hann",
            nperseg=min(256, len(seg_response)),
            detrend="linear",
            return_onesided=True,
            scaling="density",
        )
        axes[2].loglog(freq, Pxx, label=f"Gabor {f0:.3f} Hz (~{1/f0:.1f}s)")
    axes[2].set_xlabel("Fréquence (Hz)")
    axes[2].set_ylabel("PSD (m²/Hz)")
    axes[2].set_title("Spectres des réponses Gabor (Welch)")
    axes[2].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
t, g = gabor_filter(1 / 10, 12, 0.5)
plt.plot(t, g)

## Utilisation des fonctions pour Etude des résultats

### Exemple d'une comparaison via géoloc

In [ ]:
ds_WW3 = ds.sel(time="2018-09-12 00:00:00", method="nearest")

In [ ]:
dico_diff = spectre_via_geoloc(df_loc, data, "A15789", ds)

In [ ]:
data_day = dico_diff[pd.Timestamp("2019-12-27")]
res = mini_diff_day(data_day, top_n=5)

for node_id, total, diff in res:
    # print(diff)
    print(f"Node={node_id}, Somme={total:.2f}")
    print(f"Diff coefficients={diff}\n")

In [ ]:
ef = ds["ef"].sel(time="2019-12-19T00:00:00.000000000", node=7773).values
f = ds["f"].values
ef2 = np.power(10, ef) - 1e-12
plt.plot(f, ef2)
plt.xscale("log")

In [ ]:
from datetime import datetime

date1 = datetime.strptime("2019-12-17 10:00:00", "%Y-%m-%d %H:%M:%S")
date0 = datetime.strptime("2019-12-19 00:00:00.000000", "%Y-%m-%d %H:%M:%S.%f")

indice = conv_time(date1, date0, 2)
print("indice=", indice)
segment = data["pressure"].values[indice : indice + N]

freqs3, P3, P_smooth = log_smooth_spectrum(
    segment, 0.5, n_log=1000, fmin=0.03, fmax=0.953, sigma_oct=0.1
)

# Tracer le spectre de puissance
plt.figure(figsize=(8, 5))
plt.xscale("log")
plt.plot(freqs3, P_smooth, label="Spectre de puissance")
plt.plot(freqs3, P3, label="Spectre de puissance")
plt.xlabel("Fréquence [Hz]")
plt.ylabel("Puissance")
plt.title("Spectre de puissance limité entre {} et {} Hz".format(f_min, f_max))
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Spectre lissé
freqs2, P2, P_smooth = log_smooth_spectrum(
    segment, 0.5, n_log=1000, fmin=0.03, fmax=0.953, sigma_oct=0.1
)

plt.figure(figsize=(8, 5))
plt.semilogx(freqs2, 10 * np.log10(P2), label="Spectre brut")
plt.semilogx(freqs2, 10 * np.log10(P_smooth), label="Spectre lissé")
plt.legend()
plt.grid(True, which="both")
plt.show()


# Tracer le spectre de puissance
plt.figure(figsize=(8, 5))
plt.xscale("log")
plt.plot(freqs2, P_smooth, label="Spectre de puissance")
plt.plot(freqs2, P2, label="Spectre de puissance")
plt.xlabel("Fréquence [Hz]")
plt.ylabel("Puissance")
plt.title("Spectre de puissance limité entre {} et {} Hz".format(f_min, f_max))
plt.grid(True)
plt.legend()
plt.show()

### Calcul et étude de la variance 

In [ ]:
hv.extension("bokeh")  # ou 'matplotlib'

# Assurons-nous que la colonne temps est bien en datetime
data["datetime"] = pd.to_datetime(data["datetime"].values)

# Créer un Curve HoloViews
curve = hv.Curve((data["datetime"].values, data["pressure"].values), "Time", "Pressure")

# Afficher dans le notebook
hv.save(curve, "pressure_plot.html", fmt="html")


# calcul de la variance toute les 5 minutes
window_size = 150

rolling_var = (
    data["pressure"]
    .rolling(datetime=window_size)
    .construct("window")
    .var(dim="window", ddof=0)
)
# ddof=0 pour diviser par la taille de l'échantillon

In [ ]:
# Affichage de la variance
rolling_var.plot()

**Histogramme de la répartition de la variance, (pression du poisson).**

In [ ]:
import matplotlib.pyplot as plt

# Définir la limite
inf = 1
sup = 100


# Histogramme des valeurs de variance
plt.hist(rolling_var.values, bins=20, color="skyblue", edgecolor="black")
plt.xlabel("Variance")
plt.ylabel("Nombre de fenêtres")
plt.title("Histogramme des rolling variance (fenêtre=20 points)")
plt.show()

# Masquer les valeurs trop grandes pour l'histogramme
# Masquer les valeurs trop grandes ou trop petites
mask = (rolling_var.values <= sup) & (rolling_var.values >= inf)
rolling_var_filtered = rolling_var.values[mask]

# Tracer l'histogramme
plt.hist(rolling_var_filtered, bins=20, color="skyblue", edgecolor="black")
plt.xlabel("Variance")
plt.ylabel("Nombre de fenêtres")
plt.title(f"Histogramme des rolling variance (fenêtre=20 points, max={sup})")
plt.show()

### Calcul d'histogramme sur les périodes enisagées

In [ ]:
date_debut = pd.Timestamp("2019-12-17 10:00:00")
date_evt = pd.Timestamp("2020-01-18 10:00:00")
pas = 2  # échantillonnage toutes les 2 secondes
N = 3600
indice_hist = conv_time(
    date_debut, date_evt, pas, step="seconds"
)  # devrait renvoyer 300
data_hist = data["pressure"].values[indice_hist : indice_hist + N]
# Créer un histogramme
plt.figure(figsize=(8, 5))
plt.hist(
    data_hist, bins=30, color="skyblue", edgecolor="black"
)  # 30 classes par défaut
plt.xlabel("Pression")
plt.ylabel("Effectif")
plt.title("Histogramme des pressions sur deux heures")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

# calcul de la variance toute les 5 minutes
window_var = 30

import numpy as np

window = 30  # taille de la fenêtre
rolling_var = np.array(
    [
        np.var(data_hist[i : i + window], ddof=0)
        for i in range(len(data_hist) - window + 1)
    ]
)

plt.figure(figsize=(8, 5))
plt.hist(
    rolling_var, bins=30, color="skyblue", edgecolor="black"
)  # 30 classes par défaut
plt.xlabel("Pression")
plt.ylabel("varaince")
plt.title("Histogramme des variances sur deux heures")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

In [ ]:
date_debut = pd.Timestamp("2019-12-17 10:00:00")
# date_evt = [
#    pd.Timestamp("2020-01-18T00:00:00.00"),
#    pd.Timestamp("2020-01-18T10:00:00.00"),
#    pd.Timestamp("2020-01-18T20:00:00.00"),
# ]
date_evt = [
    pd.Timestamp("2020-01-24T14:00:00.00"),
    pd.Timestamp("2020-01-24T16:00:00.00"),
    pd.Timestamp("2020-01-24T18:00:00.00"),
]
pas = 2  # échantillonnage toutes les 2 secondes
N = 900  # nombre d’échantillons par segment (~30 min)

for date in date_evt:
    indice_hist = conv_time(date_debut, date, pas, step="seconds")
    data_hist = data["pressure"].values[indice_hist : indice_hist + N]

    # Axe temporel pour ce segment
    time_axis = pd.date_range(start=date, periods=N, freq=f"{pas}s")

    # ---- Création des sous-graphiques (avec layout auto) ----
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

    # ---- 1) Série temporelle ----
    axes[0, 0].plot(time_axis, data_hist, color="blue")
    axes[0, 0].set_xlabel("Temps (hh:mm)")
    axes[0, 0].set_ylabel("Pression")
    axes[0, 0].set_title("Série temporelle (pas = 2s)")
    axes[0, 0].grid(True, linestyle="--", alpha=0.6)

    # Formatter l’axe X en heures:minutes
    axes[0, 0].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    axes[0, 0].xaxis.set_major_locator(mdates.MinuteLocator(interval=5))
    for label in axes[0, 0].get_xticklabels():
        label.set_rotation(30)

    # ---- 2) Histogramme des valeurs ----
    axes[0, 1].hist(data_hist, bins=30, color="skyblue", edgecolor="black")
    axes[0, 1].set_xlabel("Pression")
    axes[0, 1].set_ylabel("Effectif")
    axes[0, 1].set_title("Histogramme des pressions")
    axes[0, 1].grid(True, linestyle="--", alpha=0.6)

    # ---- 3) Histogramme des variances glissantes ----
    window = 30
    rolling_var = np.array(
        [
            np.var(data_hist[i : i + window], ddof=0)
            for i in range(len(data_hist) - window + 1)
        ]
    )

    axes[1, 0].hist(rolling_var, bins=30, color="orange", edgecolor="black")
    axes[1, 0].set_xlabel("Variance")
    axes[1, 0].set_ylabel("Effectif")
    axes[1, 0].set_title("Histogramme des variances (fenêtre=30)")
    axes[1, 0].grid(True, linestyle="--", alpha=0.6)

    # ---- 4) Spectre brut et lissé ----
    freqs, P, P_smooth = log_smooth_spectrum(data_hist, fs=1 / pas)

    axes[1, 1].plot(freqs, P, label="Spectre brut", alpha=0.7)
    axes[1, 1].plot(freqs, P_smooth, label="Spectre lissé", lw=2)
    axes[1, 1].set_xscale("log")
    axes[1, 1].set_xlabel("Fréquence (Hz)")
    axes[1, 1].set_ylabel("Puissance")
    axes[1, 1].set_title("Spectre de puissance")
    axes[1, 1].legend()
    axes[1, 1].grid(True, which="both", linestyle="--", alpha=0.6)

    # ---- Titre général ----
    fig.suptitle(f"Analyse autour de {date}", fontsize=14)

    plt.show()

In [ ]:
date_debut = pd.Timestamp("2019-12-17 10:00:00")
date_evt = [
    pd.Timestamp("2020-01-02T18:00.00"),
]
pas = 2  # échantillonnage toutes les 2 secondes
N = 16000  # nombre d’échantillons par segment (~30 min)

for date in date_evt:
    indice_hist = conv_time(date_debut, date, pas, step="seconds")
    data_hist = data["pressure"].values[indice_hist : indice_hist + N]

    # Axe temporel pour ce segment
    time_axis = pd.date_range(start=date, periods=N, freq=f"{pas}s")

    # ---- Création des sous-graphiques (avec layout auto) ----
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

    # ---- 1) Série temporelle ----
    axes[0, 0].plot(time_axis, data_hist, color="blue")
    axes[0, 0].set_xlabel("Temps (hh:mm)")
    axes[0, 0].set_ylabel("Pression")
    axes[0, 0].set_title("Série temporelle (pas = 2s)")
    axes[0, 0].grid(True, linestyle="--", alpha=0.6)

    # Formatter l’axe X en heures:minutes
    axes[0, 0].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    axes[0, 0].xaxis.set_major_locator(mdates.MinuteLocator(interval=5))
    for label in axes[0, 0].get_xticklabels():
        label.set_rotation(30)

    # ---- 2) Histogramme des valeurs ----
    axes[0, 1].hist(data_hist, bins=30, color="skyblue", edgecolor="black")
    axes[0, 1].set_xlabel("Pression")
    axes[0, 1].set_ylabel("Effectif")
    axes[0, 1].set_title("Histogramme des pressions")
    axes[0, 1].grid(True, linestyle="--", alpha=0.6)

    # ---- 3) Histogramme des variances glissantes ----
    window = 30
    rolling_var = np.array(
        [
            np.var(data_hist[i : i + window], ddof=0)
            for i in range(len(data_hist) - window + 1)
        ]
    )

    axes[1, 0].hist(rolling_var, bins=30, color="orange", edgecolor="black")
    axes[1, 0].set_xlabel("Variance")
    axes[1, 0].set_ylabel("Effectif")
    axes[1, 0].set_title("Histogramme des variances (fenêtre=30)")
    axes[1, 0].grid(True, linestyle="--", alpha=0.6)

    # ---- 4) Spectre brut et lissé ----
    freqs, P, P_smooth = log_smooth_spectrum(data_hist, fs=1 / pas)

    axes[1, 1].plot(freqs, P, label="Spectre brut", alpha=0.7)
    axes[1, 1].plot(freqs, P_smooth, label="Spectre lissé", lw=2)
    axes[1, 1].set_xscale("log")
    axes[1, 1].set_xlabel("Fréquence (Hz)")
    axes[1, 1].set_ylabel("Puissance")
    axes[1, 1].set_title("Spectre de puissance")
    axes[1, 1].legend()
    axes[1, 1].grid(True, which="both", linestyle="--", alpha=0.6)

    # ---- Titre général ----
    fig.suptitle(f"Analyse autour de {date}", fontsize=14)

    plt.show()

In [ ]:
date_debut = pd.Timestamp("2019-12-17 10:00:00")
date_evt_temp = pd.Timestamp("2019-12-18 15:30:00")
pas = 2  # échantillonnage toutes les 2 secondes

N = 900

indice_hist = conv_time(
    date_debut, date_evt_temp, pas, step="seconds"
)  # devrait renvoyer 300
serie_temp = data["pressure"].values[indice_hist : indice_hist + N]
freqs, P, P_smooth = log_smooth_spectrum(serie_temp, 2)

plt.plot(freqs, P)
plt.plot(freqs, P_smooth)
plt.xscale("log")
plt.show()

In [ ]:
date_debut = pd.Timestamp("2019-12-17 10:00:00")
# date_evt = [
#    pd.Timestamp("2020-01-18T00:00:00.00"),
#    pd.Timestamp("2020-01-18T10:00:00.00"),
#    pd.Timestamp("2020-01-18T20:00:00.00"),
# ]
date_evt = [
    pd.Timestamp("2020-01-23T14:03:00.00"),
    pd.Timestamp("2020-01-23T16:00:00.00"),
    pd.Timestamp("2020-01-23T18:00:00.00"),
]
pas = 2  # échantillonnage toutes les 2 secondes
N = 300  # nombre d’échantillons par segment (~30 min)

for date in date_evt:
    indice_hist = conv_time(date_debut, date, pas, step="seconds")
    data_hist = data["pressure"].values[indice_hist : indice_hist + N]

    # Axe temporel pour ce segment
    time_axis = pd.date_range(start=date, periods=N, freq=f"{pas}s")

    # ---- Création des sous-graphiques (avec layout auto) ----
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

    # ---- 1) Série temporelle ----
    axes[0, 0].plot(time_axis, data_hist, color="blue")
    axes[0, 0].set_xlabel("Temps (hh:mm)")
    axes[0, 0].set_ylabel("Pression")
    axes[0, 0].set_title("Série temporelle (pas = 2s)")
    axes[0, 0].grid(True, linestyle="--", alpha=0.6)

    # Formatter l’axe X en heures:minutes
    axes[0, 0].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    axes[0, 0].xaxis.set_major_locator(mdates.MinuteLocator(interval=5))
    for label in axes[0, 0].get_xticklabels():
        label.set_rotation(30)

    # ---- 2) Histogramme des valeurs ----
    axes[0, 1].hist(data_hist, bins=100, color="skyblue", edgecolor="black")
    axes[0, 1].set_xlabel("Pression")
    axes[0, 1].set_ylabel("Effectif")
    axes[0, 1].set_title("Histogramme des pressions")
    axes[0, 1].grid(True, linestyle="--", alpha=0.6)

    # ---- 3) Histogramme des variances glissantes ----
    window = 30
    rolling_var = np.array(
        [
            np.var(data_hist[i : i + window], ddof=0)
            for i in range(len(data_hist) - window + 1)
        ]
    )

    axes[1, 0].hist(rolling_var, bins=30, color="orange", edgecolor="black")
    axes[1, 0].set_xlabel("Variance")
    axes[1, 0].set_ylabel("Effectif")
    axes[1, 0].set_title("Histogramme des variances (fenêtre=30)")
    axes[1, 0].grid(True, linestyle="--", alpha=0.6)

    # ---- 4) Spectre brut et lissé ----
    freqs, P, P_smooth = log_smooth_spectrum(data_hist, fs=1 / pas)

    axes[1, 1].plot(freqs, P, label="Spectre brut", alpha=0.7)
    axes[1, 1].plot(freqs, P_smooth, label="Spectre lissé", lw=2)
    axes[1, 1].set_xscale("log")
    axes[1, 1].set_xlabel("Fréquence (Hz)")
    axes[1, 1].set_ylabel("Puissance")
    axes[1, 1].set_title("Spectre de puissance")
    axes[1, 1].legend()
    axes[1, 1].grid(True, which="both", linestyle="--", alpha=0.6)

    # ---- Titre général ----
    fig.suptitle(f"Analyse autour de {date}", fontsize=14)

    plt.show()

## Bricolage

In [ ]:
# Supposons que ton Dataset s'appelle ds
ID_cible = 15789
date_cible = pd.Timestamp("2020-01-19")

# Sélection avec .sel
mini_ds = ds_loc.sel(ID=ID_cible, time=date_cible)

(mini_ds)

In [ ]:
date_debut = data["datetime"].values[0]
N = 600  # nombre de points (= 20 minutes)
dt = 2  # pas de temps en secondes
freq_WW3 = ds["f"].values
lat, lon = mini_ds["lat"], mini_ds["lon"]
# Début et fin
start_time = pd.Timestamp("2020-01-19 08:00:00")
end_time = pd.Timestamp("2020-01-19 18:00:00")
dico_total = {}
# Création de l'échelle temporelle toutes les 20 minutes
time_axis = pd.date_range(
    start=start_time, end=end_time, freq="20min"
)  # "min" = minutes

for time in time_axis:
    dico_diff = {}
    data_WW3_time = ds.sel(time=time, method="nearest")
    # print(date_debut,time)
    indice = conv_time(date_debut, time, dt, step="seconds")
    segment = data["pressure"].values[indice : indice + N]
    freqs2, P2, P_smooth = log_smooth_spectrum(
        segment, 0.5, n_log=1000, fmin=0.03, fmax=0.953, sigma_oct=0.1
    )
    liste_node, index_nodes = stations_proches(data_WW3_time, lat, lon)
    # print("Les valeurs des node:", liste_node.node.values)
    # if i < 3:
    # print(liste_node)
    values_ef = liste_node["ef"].values

    # Transformer en une liste de spectres (un par nœud)
    list_spectres = [values_ef[:, i].tolist() for i in range(values_ef.shape[1])]

    # print("ICIIIIIIIIIIIIIIIIIIIIIII", list_spectres)
    for j in range(len(list_spectres)):
        # print(len(values_ef),len(values_ef[i]),print(len(freq_WW3)))
        # print((values_ef),(values_ef[i]))
        diff, freq_diff = calcul_diff(P2, list_spectres[j], freqs2, freq_WW3)
        dico_diff[index_nodes[j]] = [diff, freq_diff, list_spectres[j]]
    dico_total[time] = dico_diff

In [ ]:
print(time_axis)
print(list(dico_diff.keys())[:10])  # affiche les 10 premières clés

In [ ]:
data_day = dico_total[pd.Timestamp("2020-01-19 10:00:00")]
# print(data_day)
res = mini_diff_day(data_day, top_n=5)

for node_id, total, diff in res:
    # print(diff)
    print(f"Node={node_id}, Somme={total:.2f}")
    print(f"Diff coefficients={diff}\n")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

date_debut = pd.Timestamp("2019-12-17 10:00:00")
date_evt = [
    pd.Timestamp(
        "2020-01-18T15:20:00.00",
    )
]
pas = 2  # échantillonnage toutes les 2 secondes
N = 600  # nombre d’échantillons (~20 min ici)

for date in date_evt:
    indice_hist = conv_time(date_debut, date, pas, step="seconds")
    data_hist = data["pressure"].values[indice_hist : indice_hist + N]

    # Axe temporel
    time_axis = pd.date_range(start=date, periods=N, freq=f"{pas}s")

    # ---- 1) Série temporelle ----
    plt.figure(figsize=(12, 5))
    plt.plot(time_axis, data_hist, color="blue")
    plt.xlabel("Temps")
    plt.ylabel("Pression")
    plt.title(f"Série temporelle de pression à partir de {date}")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.xticks(rotation=30)
    plt.show()

    # ---- 2) Spectre brut et lissé ----
    freqs, P, P_smooth = log_smooth_spectrum(data_hist, fs=1 / pas)

    plt.figure(figsize=(12, 5))
    plt.plot(freqs, P, label="Spectre brut", alpha=0.7)
    plt.plot(freqs, P_smooth, label="Spectre lissé", lw=2)
    plt.xscale("log")
    plt.xlabel("Fréquence (Hz)")
    plt.ylabel("Puissance")
    plt.title("Spectre de puissance")
    plt.legend()
    plt.grid(True, which="both", linestyle="--", alpha=0.6)
    plt.show()

In [ ]:
def plot_spectres(data_day, node_id, freqs, P, P_smooth, freq_WW3):

    # Extraction des données
    diff = data_day[node_id][0]
    freq = data_day[node_id][1]
    ef = data_day[node_id][2]

    # Conversion ef → puissance (WW3 en linéaire)
    ef2 = np.power(10, ef) + 1e-12

    # === Restriction des spectres poisson (P, P_smooth) au domaine WW3 ===
    fmin, fmax = freq_WW3.min(), freq_WW3.max()
    mask = (freqs >= fmin) & (freqs <= fmax)

    freqs_common = freqs[mask]
    P_common = P[mask]
    P_smooth_common = P_smooth[mask]

    # --- Tracé ---
    plt.figure(figsize=(10, 6))

    plt.plot(freq_WW3, ef2, label="ef2 (WW3)", lw=2)
    plt.plot(freq, diff, label="Diff (poisson - WW3)", lw=2, color="pink")
    plt.plot(freqs_common, P_common, label="Spectre brut", alpha=0.7, color="grey")
    plt.plot(freqs_common, P_smooth_common, label="Spectre lissé", lw=2, color="red")

    plt.xscale("log")
    plt.xlabel("Fréquence (Hz)")
    plt.ylabel("Puissance")
    plt.title(f"Spectres pour le node {node_id}")
    plt.legend()
    plt.grid(True, which="both", linestyle="--", alpha=0.6)
    plt.show()


node_id = 138279
plot_spectres(data_day, node_id, freqs, P, P_smooth, freq_WW3)